In [1]:
# Notebook 2 — Celda 1: Cargar activaciones de la Etapa 1
import torch, numpy as np, json, os, yaml, re
from google.colab import drive
drive.mount('/content/drive')

BASE       = '/content/drive/MyDrive/nla_pipeline'
CHECKPOINT_AV = f'{BASE}/checkpoints/nla_av'
DIR_ACT    = f'{BASE}/activaciones'
DIR_VERB   = f'{BASE}/verbalizaciones'
os.makedirs(DIR_VERB, exist_ok=True)

# Cargar activaciones y metadatos
activaciones = np.load(f'{DIR_ACT}/activaciones_L20.npy')   # [T, 3584]
with open(f'{DIR_ACT}/metadatos.json') as f:
    meta = json.load(f)

print(f'Texto original: {meta["texto"]}')
print(f'Tokens totales: {meta["n_tokens"]}')
print(f'Posiciones a verbalizar: {meta["posiciones"]}')
print(f'Forma tensor activaciones: {activaciones.shape}')



Mounted at /content/drive
Texto original: The Eiffel Tower was built in Paris during the 1889 World Fair.
Tokens totales: 48
Posiciones a verbalizar: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
Forma tensor activaciones: (48, 3584)


In [2]:
# Notebook 2 — Celda 2: Leer configuración del sidecar
with open(f'{CHECKPOINT_AV}/nla_meta.yaml') as f:
    nla_meta = yaml.safe_load(f)

# Extraer parámetros clave
D_MODEL          = nla_meta['d_model']                          # 3584
INJECTION_SCALE  = nla_meta['extraction']['injection_scale']    # 150.0
INJECTION_CHAR   = nla_meta['tokens']['injection_char']         # '㈎'
INJECTION_TOK_ID = nla_meta['tokens']['injection_token_id']     # 149705
LEFT_NEIGHBOR    = nla_meta['tokens']['injection_left_neighbor_id']
RIGHT_NEIGHBOR   = nla_meta['tokens']['injection_right_neighbor_id']
PROMPT_TEMPLATE  = nla_meta['prompt_templates']['av']

print(f'd_model:          {D_MODEL}')
print(f'injection_scale:  {INJECTION_SCALE}')
print(f'injection_char:   {repr(INJECTION_CHAR)}')
print(f'injection_tok_id: {INJECTION_TOK_ID}')
print(f'Prompt template (primeros 80 chars): {PROMPT_TEMPLATE[:80]}...')



d_model:          3584
injection_scale:  150.0
injection_char:   '㈎'
injection_tok_id: 149705
Prompt template (primeros 80 chars): You are a meticulous AI researcher conducting an important investigation into ac...


In [4]:
pip install -U bitsandbytes>=0.46.1

In [5]:
# Notebook 2 — Celda 3: Cargar AV
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from safetensors import safe_open
import math

# Detectar VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20
print(f'VRAM: {vram_gb:.1f} GB → Modo: {"8-bit" if MODO_8BIT else "bfloat16"}')

# Cargar tokenizer del AV
tok_av = AutoTokenizer.from_pretrained(CHECKPOINT_AV, trust_remote_code=True)

# Verificar que el carácter de inyección tokeniza correctamente
ids_test = tok_av.encode(INJECTION_CHAR, add_special_tokens=False)
assert ids_test == [INJECTION_TOK_ID], \
    f'ERROR: {INJECTION_CHAR} → {ids_test}, esperado [{INJECTION_TOK_ID}]'
print(f'✓ Carácter de inyección verificado: {repr(INJECTION_CHAR)} → {ids_test}')

# Cargar modelo AV
print('Cargando AV...')
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
if MODO_8BIT:
    av_model = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AV, quantization_config=quantization_config, device_map='auto',
        trust_remote_code=True,
    )
else:
    av_model = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AV, torch_dtype=torch.bfloat16,
        device_map='cuda:0', trust_remote_code=True,
    )
av_model.eval()
print(f'✓ AV cargado. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Cargar SOLO el peso del embedding (lazy load — no cargar todo el modelo de nuevo)
# Necesitamos el embedding para construir inputs_embeds
DTYPE_EMB = torch.bfloat16 if not MODO_8BIT else torch.float16

def cargar_embedding_rapido(checkpoint_dir, dtype):
    '''Carga solo la capa de embedding sin materializar el modelo completo.'''
    import json
    index_path = f'{checkpoint_dir}/model.safetensors.index.json'
    if os.path.exists(index_path):
        weight_map = json.load(open(index_path))['weight_map']
        key = [k for k in weight_map if k.endswith('embed_tokens.weight')][0]
        shard = f'{checkpoint_dir}/{weight_map[key]}'
    else:
        shard = f'{checkpoint_dir}/model.safetensors'
        with safe_open(shard, framework='pt') as f_:
            key = [k for k in f_.keys() if k.endswith('embed_tokens.weight')][0]
    with safe_open(shard, framework='pt') as f_:
        weight = f_.get_tensor(key).to(dtype)
    emb = torch.nn.Embedding(*weight.shape, _weight=weight)
    emb.requires_grad_(False)
    return emb.eval()

embed_layer = cargar_embedding_rapido(CHECKPOINT_AV, DTYPE_EMB)
# Para Qwen, embed_scale = 1.0 (no hay escala adicional)
EMBED_SCALE = 1.0
print('✓ Capa de embedding cargada')



VRAM: 15.6 GB → Modo: 8-bit
✓ Carácter de inyección verificado: '㈎' → [149705]
Cargando AV...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ AV cargado. VRAM usada: 8.7 GB
✓ Capa de embedding cargada


In [15]:
# Notebook 2 — Celda 4: Función de verbalización

def verbalizar(v_raw, temperatura=1.0, max_nuevos_tokens=200):
    '''
    Dada una activación v_raw de forma [3584], retorna
    una descripción en lenguaje natural generada por el AV.

    Pasos internos:
    1. Tokenizar el prompt con el carácter especial ㈎
    2. Construir la matriz de embeddings
    3. Reemplazar la fila de ㈎ con v_raw reescalado
    4. Llamar model.generate(inputs_embeds=...)
    5. Extraer el texto de <explanation>...</explanation>
    '''
    # Paso 1: Tokenizar el prompt exacto del sidecar
    contenido = PROMPT_TEMPLATE.format(injection_char=INJECTION_CHAR)

    # First, get the formatted string with the chat template applied (tokenize=False)
    formatted_string = tok_av.apply_chat_template(
        [{'role': 'user', 'content': contenido}],
        tokenize=False, # Get the raw string output
        add_generation_prompt=True,
    )

    # Now, encode this string into a list of token IDs
    # add_special_tokens=False because apply_chat_template already handles special tokens in the string
    input_ids = tok_av.encode(formatted_string, add_special_tokens=False)

    # Paso 2: Construir embeddings [1, T, 3584]
    ids_t = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        embeds = (embed_layer(ids_t) * EMBED_SCALE).float()   # [1, T, 3584]

    # Paso 3: Rescalar e inyectar v_raw
    # injection_scale = 150: norma L2 a la que el AV fue entrenado
    v = torch.as_tensor(v_raw, dtype=torch.float32)
    norma = v.norm().clamp_min(1e-12)
    v_scaled = v * (INJECTION_SCALE / norma)   # reescalar a norma 150

    # Encontrar la posición de inyección verificando vecinos
    inyectado = False
    for p in range(1, len(input_ids) - 1):
        if (input_ids[p]   == INJECTION_TOK_ID and
            input_ids[p-1] == LEFT_NEIGHBOR and
            input_ids[p+1] == RIGHT_NEIGHBOR):
            embeds[0, p] = v_scaled
            inyectado = True
            break
    assert inyectado, 'No se encontró posición de inyección — revisar prompt template'

    # Paso 4: Generar con inputs_embeds (sin input_ids)
    # model.generate() acepta inputs_embeds directamente
    device = next(av_model.parameters()).device
    embeds_dev = embeds.to(device)
    if MODO_8BIT:
        # Ensure input embeddings are bfloat16 to match model's compute dtype in 8-bit mode
        embeds_dev = embeds_dev.bfloat16()

    with torch.no_grad():
        tokens_gen = av_model.generate(
            inputs_embeds=embeds_dev,
            max_new_tokens=max_nuevos_tokens,
            temperature=temperatura,
            do_sample=True,
            pad_token_id=tok_av.eos_token_id,
        )

    # Paso 5: Decodificar y extraer <explanation>
    texto_gen = tok_av.decode(tokens_gen[0], skip_special_tokens=False)
    m = re.search(r'<explanation>\s*(.*?)\s*</explanation>', texto_gen, re.DOTALL)
    if m:
        return m.group(1).strip()
    else:
        # Si no hay tags: generación truncada → aumentar max_nuevos_tokens
        print('AVISO: no se encontraron tags <explanation>. Retornando texto crudo.')
        return texto_gen

print('✓ Función verbalizar() lista')

✓ Función verbalizar() lista


In [16]:
# Notebook 2 — Celda 5: Verbalizar y guardar
resultados_verbalizacion = []

print(f'Verbalizando {len(meta["posiciones"])} tokens...')
print('(~10-30 segundos por token)\n')

for i, pos in enumerate(meta['posiciones']):
    v_raw = activaciones[pos]   # [3584]
    norma = float(np.linalg.norm(v_raw))

    print(f'Token [{pos}] — norma={norma:.1f}', end=' ... ')
    descripcion = verbalizar(v_raw, temperatura=1.0, max_nuevos_tokens=200)
    print(f'✓')
    print(f'  → {descripcion[:80]}...')

    resultados_verbalizacion.append({
        'posicion': pos,
        'norma_l2': norma,
        'descripcion': descripcion,
    })

# Guardar resultados
import json
ruta_verb = f'{DIR_VERB}/verbalizaciones.json'
with open(ruta_verb, 'w', encoding='utf-8') as f:
    json.dump(resultados_verbalizacion, f, ensure_ascii=False, indent=2)

print(f'\n✓ {len(resultados_verbalizacion)} verbalizaciones guardadas en Drive')
print(f'  Ruta: {ruta_verb}')
print('\n🎉 Etapa 2 completada. Continúa con Notebook_3_ar_score.ipynb')



Verbalizando 38 tokens...
(~10-30 segundos por token)

Token [10] — norma=113.4 ... 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


✓
  → Structured AI-generated article with "Chinese Character" format and "a cute cat"...
Token [11] — norma=109.5 ... ✓
  → Chinese language model format with structured "definition" pattern establishing ...
Token [12] — norma=106.5 ... ✓
  → AI system prompt format with "I am an AI assistant created by Alibaba Cloud." si...
Token [13] — norma=112.8 ... ✓
  → Formal AI system introduction with "ChatGPT" and "Human-Oriented" framing, sugge...
Token [14] — norma=107.4 ... ✓
  → Structured AI product description format with "assistant" identity and capabilit...
Token [15] — norma=107.3 ... ✓
  → AI prompt structure with "Name: Qwen is a large language model" suggesting ChatG...
Token [16] — norma=118.0 ... ✓
  → Structured AI model prompt format with "Assistant: " label following a standard ...
Token [17] — norma=110.6 ... ✓
  → Formal AI platform UI context with a greeting ("I am Alpha GPT, a large language...
Token [18] — norma=100.1 ... ✓
  → Chinese language model format with "Answer